# 23CSE301 Machine Learning — Capstone Project
## Clustering Track — Part 1
### Team TASKMASTERS (Team 16)

**Dataset:** WTA Grand Slam Matches (2018–2022)

**Purpose of this notebook:**  
This is the first GitHub-ready part of the clustering track. It covers:

1. Dataset loading and basic audit
2. Exploratory analysis of match statistics
3. Selection of numerical clustering features
4. Missing-value handling and standardisation
5. K-Means clustering
6. Elbow method and Silhouette Score
7. PCA-based 2D cluster visualisation

> **Note:** Hierarchical clustering, Davies–Bouldin Index, Calinski–Harabasz Index, dendrogram, and t-SNE can be added as the next part.


### Dataset source

The official SCORE repository describes this dataset as **2,413 WTA Grand Slam matches from 2018–2022**, with 38 columns covering tournament information, player information, and winner/loser match statistics.

Official source: https://data.scorenetwork.org/tennis/wta-grand-slam-matches.html

The clustering idea is to group matches according to observed winner/loser match statistics and later examine how the resulting clusters relate to **court surface** or **Grand Slam tournament**.


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


## 1. Load the dataset

In [ ]:
# Try the local GitHub data path first.
# If the CSV is not present locally, load it directly from the official SCORE source.

DATA_PATH = "../data/wta-grand-slam-matches-2018to2022.csv"
DATA_URL = "https://data.scorenetwork.org/data/wta-grand-slam-matches-2018to2022.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print("Loaded local dataset:", DATA_PATH)
except FileNotFoundError:
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from the official SCORE URL.")

print("Shape:", df.shape)
df.head()


## 2. Basic data audit

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Target-free clustering setup

Clustering is **unsupervised**, so there is no target variable used to train the clusters.

For this first part, we use numerical winner/loser match statistics. We deliberately do not use:

- `winner_name`
- `loser_name`
- `score`
- `tourney_name`
- `surface`
- `round`

`surface` and `tourney_name` are retained separately for later interpretation of the clusters, rather than being used to create the clusters themselves.


In [ ]:
# Match-statistics features supported by the SCORE dataset.
match_stat_features = [
    "minutes",
    "w_ace", "l_ace",
    "w_df", "l_df",
    "w_svpt", "l_svpt",
    "w_1stIn", "l_1stIn",
    "w_1stWon", "l_1stWon",
    "w_2ndWon", "l_2ndWon",
    "w_SvGms", "l_SvGms",
    "w_bpSaved", "l_bpSaved",
    "w_bpFaced", "l_bpFaced"
]

available_features = [c for c in match_stat_features if c in df.columns]
missing_expected = [c for c in match_stat_features if c not in df.columns]

print("Available clustering features:", len(available_features))
print(available_features)

if missing_expected:
    print("\nExpected columns not found:", missing_expected)

X_raw = df[available_features].copy()
X_raw.describe().T


## 4. Preprocessing

In [ ]:
# Median imputation + StandardScaler.
# The preprocessing is kept inside a Pipeline so the same transformations
# are applied consistently before clustering.

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_scaled = preprocessor.fit_transform(X_raw)

print("Original feature matrix:", X_raw.shape)
print("Preprocessed feature matrix:", X_scaled.shape)
print("Any missing values after preprocessing:", np.isnan(X_scaled).sum())


### Why StandardScaler?

The clustering features have different units and ranges. For example, match duration is measured in minutes while aces and double faults are counts.

Standardisation puts the numerical features on a comparable scale before distance-based clustering.

### Why median imputation?

Some match-statistic columns can contain missing values. Median imputation provides a simple robust replacement without deleting complete match rows.


## 5. K-Means — choose a suitable number of clusters

In [ ]:
# Evaluate K-Means for several values of K.
k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

k_results = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores
})

display(k_results.round(4))


In [ ]:
# Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(k_results["K"], k_results["Inertia"], marker="o")
plt.title("K-Means Elbow Curve")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()

# Silhouette curve
plt.figure(figsize=(8, 5))
plt.plot(k_results["K"], k_results["Silhouette Score"], marker="o")
plt.title("Silhouette Score vs Number of Clusters")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.xticks(list(k_values))
plt.tight_layout()
plt.show()


## 6. Fit K-Means

For this first part, the value of **K is selected using the Silhouette Score**, while the elbow curve is used as an additional diagnostic.

The code below automatically selects the K with the highest Silhouette Score among the tested values.


In [ ]:
best_k = int(
    k_results.loc[k_results["Silhouette Score"].idxmax(), "K"]
)

kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

cluster_labels = kmeans_model.fit_predict(X_scaled)

clustered_df = df.copy()
clustered_df["cluster"] = cluster_labels

print("Selected K:", best_k)
print("\nCluster sizes:")
display(clustered_df["cluster"].value_counts().sort_index().to_frame("count"))


## 7. Cluster profiles

In [ ]:
# Mean match statistics for each cluster.
cluster_profile = (
    clustered_df.groupby("cluster")[available_features]
    .mean()
    .round(2)
)

display(cluster_profile)


## 8. PCA visualisation

PCA reduces the standardised feature space to two dimensions for visualisation.

The PCA plot is **not** the space used to fit K-Means; K-Means was fitted on the full preprocessed feature matrix.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "cluster": cluster_labels
})

plt.figure(figsize=(9, 6))

for cluster_id in sorted(pca_df["cluster"].unique()):
    subset = pca_df[pca_df["cluster"] == cluster_id]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        alpha=0.55,
        s=25,
        label=f"Cluster {cluster_id}"
    )

plt.title("K-Means Clusters — PCA 2D Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

print(
    "Explained variance by PC1 and PC2:",
    np.round(pca.explained_variance_ratio_, 4)
)


## 9. Quick interpretation

At this stage, the clusters represent groups of WTA Grand Slam matches with similar **winner/loser match-statistic profiles**.

The cluster labels themselves do not have an inherent meaning such as "good" or "bad". We interpret them by comparing their average match statistics and, in the next part, by checking their relationship with:

- court surface (`surface`)
- Grand Slam tournament (`tourney_name`)

### Next clustering sections to add

- Hierarchical/Agglomerative Clustering
- Davies–Bouldin Index
- Calinski–Harabasz Index
- Dendrogram
- PCA visualisation for hierarchical clustering
- t-SNE visualisation for at least one algorithm
- Final comparison table


### Reproducibility

- Random state: **42**
- Missing numerical values: **median imputation**
- Scaling: **StandardScaler**
- Clustering algorithm in this part: **K-Means**
